In [ ]:
!pip install numpy ipympl librosa

In [ ]:
import numpy as np
import librosa as lr
from numpy import pi
from scipy import signal
import random
import matplotlib.pyplot as plt
%matplotlib inline

# Вариант 6

### Задание

1. Ознакомиться с теоретической частью.

1. Для тестовых музыкальных файлов реализовать мел-спектраграмму:

    - [ ] реализовать алгоритм вычисления мел-спектраграммы
  
    - [x] реализовать мел-спектраграмму с помощью готовых библиотек
  
    - объяснить полученный результат
  
1. Для тестовых музыкальных файлов вычислить спектральные признаки аудиосигнала:

    - самостоятельно:
  
        - [ ] частота пересечения нуля
      
        - [ ] спектральная ширина ширина
     
    - с помощью библиотек:
  
        - [ ] любой из методы на выбор
     
        - [ ] любой из методы на выбор
     
        - [ ] любой из методы на выбор

    - объяснить полученный результат
  
1. Написать функцию, которая будет смешивать чистый голос и шум по `SNR = 0,3..15 дБ`.

1. Сравнить методы оценки качества звука:

    - самостоятельно:
  
        - [ ] SNR
     
        - [ ] SDR
     
    - с помощью библиотек:
  
        - [ ] SI-SDR
     
        - [ ] PESQ
     
        - [ ] NISQA
     
        - [ ] DNSMOS
     
    - [ ] вывести в виде таблицы:
  
      тестовый файл | объективные оценки (SNR, SDR, etc.) | субъективная оценка
      ---           |---                                  |---
  
1. Прогнать через шумоподавление:

    - [ ] установить модель шумоподавления (DeepFilterNet2 или более актуальную)
  
    - [ ] повторить тесты из п.5
      

In [ ]:
samples, sr = lr.load('2sine.wav', sr=None)

# МЕЛ-Спектрограмма

In [ ]:
def show_lr_mel_spectr(samples, mel_count=256):
    mel_spectr = lr.feature.melspectrogram(y=samples, sr=sr, n_mels=mel_count)
    mel_spectr_db = lr.power_to_db(mel_spectr, top_db=None)
    
    plt.figure(figsize=(12, 4))
    lr.display.specshow(mel_spectr_db, sr=sr, x_axis='time', y_axis='mel', cmap="magma")
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

show_lr_mel_spectr(samples)

In [ ]:
def show_our_mel_spectr(samples):
    def mel(hz): return 2595 * np.log10(1+hz/700)
    def unmel(mel): return 700 * (np.pow(10, mel/2595) - 1)
    def db(x): return 10 * np.log10(x/np.max(x))

    def mel_banks(count: int, freqs):
        f_min_mel = mel(min(freqs))
        f_max_mel = mel(max(freqs))
        f_mel_step = (f_max_mel - f_min_mel) / count

        banks = []
        for f_mel in np.arange(f_min_mel + f_mel_step, f_max_mel, f_mel_step):
            f_mid, f_start, f_end = unmel(f_mel), unmel(f_mel-f_mel_step), unmel(f_mel+f_mel_step)
            mid_sample, start_sample, end_sample = np.abs(freqs-f_mid).argmin(), np.abs(freqs-f_start).argmin(), np.abs(freqs-f_end).argmin()

            tri_leftwidth, tri_rightwidth = mid_sample-start_sample, end_sample-mid_sample
            tri = np.concatenate((
                np.arange(0, 1, 1/tri_leftwidth),
                np.flip(np.arange(0, 1, 1/tri_rightwidth))
            ))
            banks.append(np.pad(tri, pad_width=(start_sample, len(freqs)-end_sample), constant_values=0))
        
        return banks

    
        
    # https://www.youtube.com/watch?v=-Yxj3yfvY-4
    freqs, times, stft = signal.stft(samples, sr, nperseg=1024) # TODO? probably implement ourselves
    ampls = db(np.pow(np.abs(stft), 2))

    mel_banks(10, freqs)
    
    # TODO! incomplete - does not convert to mel, just plots on a mel-scale
    mel_freqs = mel(freqs)

    plt.figure(figsize=(12, 4))
    plt.pcolormesh(times, mel_freqs, ampls, cmap="magma")
    plt.ylabel('Hz')
    plt.xlabel('Time')
    plt.colorbar(format="%+2.0f dB")
    plt.tight_layout()
    plt.show()

show_our_mel_spectr(samples)